In [1]:
from __future__ import annotations

from collections import deque
from bisect import bisect_left, insort
from typing import Deque, Dict, List, Tuple, Optional


class StreamProcessor:
    def __init__(self):
        """
        Internal state:
          - self.orders: seq -> (side, price, remaining_volume) for RESTING orders only
          - self.level_queues: side -> price -> deque[(seq, remaining_volume)]
              * we allow "lazy" deletion: cancellations/fully-filled orders are removed from
                self.orders and their remaining set to 0 in-place in the queue when encountered.
          - self.level_vol: side -> price -> total resting volume at that price (aggregate)
          - self.bid_prices: sorted ascending list of bid prices (best bid is last)
          - self.ask_prices: sorted ascending list of ask prices (best ask is first)
        """
        self.orders: Dict[int, Tuple[str, float, int]] = {}

        self.level_queues: Dict[str, Dict[float, Deque[Tuple[int, int]]]] = {
            "BUY": {},
            "SELL": {},
        }
        self.level_vol: Dict[str, Dict[float, int]] = {
            "BUY": {},
            "SELL": {},
        }

        self.bid_prices: List[float] = []
        self.ask_prices: List[float] = []

    # ----------------------------
    # Helpers for price-level sets
    # ----------------------------
    def _price_list(self, side: str) -> List[float]:
        return self.bid_prices if side == "BUY" else self.ask_prices

    def _ensure_price_level(self, side: str, price: float) -> None:
        """Ensure price is present in the sorted price list for that side."""
        plist = self._price_list(side)
        i = bisect_left(plist, price)
        if i == len(plist) or plist[i] != price:
            insort(plist, price)

    def _remove_price_level_if_empty(self, side: str, price: float) -> None:
        """Remove price from structures if its aggregate volume is zero."""
        if self.level_vol[side].get(price, 0) > 0:
            return

        # Remove from aggregate + queue maps
        self.level_vol[side].pop(price, None)
        self.level_queues[side].pop(price, None)

        # Remove from sorted price list
        plist = self._price_list(side)
        i = bisect_left(plist, price)
        if i < len(plist) and plist[i] == price:
            plist.pop(i)

    def _add_resting(self, seq: int, side: str, price: float, volume: int) -> None:
        """Add a new RESTING order."""
        if volume <= 0:
            return

        self.orders[seq] = (side, price, volume)

        if price not in self.level_queues[side]:
            self.level_queues[side][price] = deque()
        self.level_queues[side][price].append((seq, volume))

        self.level_vol[side][price] = self.level_vol[side].get(price, 0) + volume
        self._ensure_price_level(side, price)

    def _best_price(self, side: str) -> Optional[float]:
        """Return best price for side among positive-volume levels, else None."""
        plist = self._price_list(side)
        if not plist:
            return None
        return plist[-1] if side == "BUY" else plist[0]

    # ----------------------------
    # Matching engine core
    # ----------------------------
    def _match_against_level(self, taker_remaining: int, maker_side: str, price: float) -> int:
        """
        Consume volume from maker_side at given price, FIFO within that price.
        Returns remaining taker volume after matching this level as much as possible.
        """
        if taker_remaining <= 0:
            return 0

        q = self.level_queues[maker_side].get(price)
        if not q:
            return taker_remaining

        while taker_remaining > 0 and q and self.level_vol[maker_side].get(price, 0) > 0:
            seq, rem = q[0]

            # Lazy skip if order no longer exists or has zero remaining
            if rem <= 0 or seq not in self.orders:
                q.popleft()
                continue

            maker_side0, maker_price0, maker_rem0 = self.orders[seq]
            # If mapping disagrees (shouldn't), treat as invalid and drop queue head
            if maker_side0 != maker_side or maker_price0 != price or maker_rem0 <= 0:
                q.popleft()
                self.orders.pop(seq, None)
                continue

            trade = maker_rem0 if maker_rem0 < taker_remaining else taker_remaining

            # Update maker order + aggregates
            new_maker_rem = maker_rem0 - trade
            taker_remaining -= trade

            self.level_vol[maker_side][price] -= trade

            if new_maker_rem == 0:
                # fully filled
                self.orders.pop(seq, None)
                q.popleft()
            else:
                # partially filled: update order map and queue head in place
                self.orders[seq] = (maker_side, price, new_maker_rem)
                q[0] = (seq, new_maker_rem)

        # If level depleted, clean it up
        if self.level_vol[maker_side].get(price, 0) <= 0:
            self._remove_price_level_if_empty(maker_side, price)

        return taker_remaining

    def add_message(self, message: dict) -> None:
        action = message.get("action")
        seq = int(message["seq"])

        # Handle cancellation (seq guaranteed to exist in the book as a RESTING order)
        if action == "CANCEL":
            side, price, rem = self.orders.pop(seq)  # guaranteed present
            if rem > 0:
                # Adjust aggregate volume; queue entry is lazily removed later
                self.level_vol[side][price] = self.level_vol[side].get(price, 0) - rem
                if self.level_vol[side][price] <= 0:
                    self._remove_price_level_if_empty(side, price)
            return

        side = message["side"]
        price = float(message["price"])
        volume = int(message["volume"])
        if volume <= 0:
            return

        remaining = volume

        if side == "BUY":
            # BUY matches against SELL (asks) starting from best ask while ask_price <= buy_price
            while remaining > 0:
                best_ask = self._best_price("SELL")
                if best_ask is None or best_ask > price:
                    break
                remaining = self._match_against_level(remaining, "SELL", best_ask)

            # Rest any leftover on BUY side
            if remaining > 0:
                self._add_resting(seq, "BUY", price, remaining)

        elif side == "SELL":
            # SELL matches against BUY (bids) starting from best bid while bid_price >= sell_price
            while remaining > 0:
                best_bid = self._best_price("BUY")
                if best_bid is None or best_bid < price:
                    break
                remaining = self._match_against_level(remaining, "BUY", best_bid)

            # Rest any leftover on SELL side
            if remaining > 0:
                self._add_resting(seq, "SELL", price, remaining)
        else:
            raise ValueError(f"Unknown side: {side!r}")

    def get_mid_price(self) -> float:
        best_bid = self._best_price("BUY")
        best_ask = self._best_price("SELL")
        if best_bid is None or best_ask is None:
            return 0.0
        return (best_bid + best_ask) / 2.0

    def get_book_depth(self) -> Dict[str, List[Tuple[float, int]]]:
        bids: List[Tuple[float, int]] = []
        asks: List[Tuple[float, int]] = []

        # BIDS descending
        for p in reversed(self.bid_prices):
            v = self.level_vol["BUY"].get(p, 0)
            if v > 0:
                bids.append((p, v))

        # ASKS ascending
        for p in self.ask_prices:
            v = self.level_vol["SELL"].get(p, 0)
            if v > 0:
                asks.append((p, v))

        return {"BIDS": bids, "ASKS": asks}


In [2]:
import pytest
from solution import StreamProcessor
import sys


if sys.version_info.major != 3 or sys.version_info.minor != 11:
    print("-" * 60)
    print("ENVIRONMENT ERROR: Python 3.11 is required for this assessment.")
    print(f"You are currently running: {sys.version.split()[0]}")
    print("Please switch to Python 3.11 to avoid compatibility issues with the grader.")
    print("-" * 60)
    sys.exit(1)


def test_price_priority_execution():
    """
    Requirement 2: Execution always occurs at the most favorable price.
    Incoming BUY must deplete the lowest SELL (Ask) prices first.
    """
    sp = StreamProcessor()
    # Populate two sell levels (Asks)
    sp.add_message({"seq": 1, "side": "SELL", "price": 10.0, "volume": 10})
    sp.add_message({"seq": 2, "side": "SELL", "price": 11.0, "volume": 10})

    # Buy order that crosses both levels.
    # Logic: Should take 10 units at 10.0, then 5 units at 11.0.
    sp.add_message({"seq": 3, "side": "BUY", "price": 11.0, "volume": 15})

    depth = sp.get_book_depth()
    # All volume at 10.0 should be gone, 5 units remain at 11.0
    assert depth["ASKS"] == [(11.0, 5)]
    assert depth["BIDS"] == []


def test_mid_price_calculation():
    """Requirement: Mid-price is the average of the best bid and best ask."""
    sp = StreamProcessor()
    sp.add_message({"seq": 1, "side": "BUY", "price": 99.0, "volume": 10})
    sp.add_message({"seq": 2, "side": "SELL", "price": 101.0, "volume": 10})

    # (99.0 + 101.0) / 2 = 100.0
    assert sp.get_mid_price() == 100.0


def test_order_cancellation():
    """Requirement: CANCEL removes volume associated with a specific sequence identifier."""
    sp = StreamProcessor()
    sp.add_message({"seq": 1, "side": "BUY", "price": 50.0, "volume": 100})
    sp.add_message({"seq": 2, "side": "BUY", "price": 50.0, "volume": 50})

    # Verify aggregate volume at the level is 150
    assert sp.get_book_depth()["BIDS"] == [(50.0, 150)]

    # Cancel the first order.
    # Note: We include side/price/volume to match the dict schema, even if action is CANCEL.
    sp.add_message(
        {"seq": 1, "side": "BUY", "price": 50.0, "volume": 100, "action": "CANCEL"}
    )

    # Remaining volume should only be from seq 2
    assert sp.get_book_depth()["BIDS"] == [(50.0, 50)]


def test_incomplete_book_mid_price():
    """Requirement: Return 0.0 if the book lacks orders on either side."""
    sp = StreamProcessor()
    sp.add_message({"seq": 1, "side": "BUY", "price": 10.0, "volume": 5})
    # ASKS side is empty, so mid-price must be 0.0
    assert sp.get_mid_price() == 0.0


def test_book_depth_sorting():
    """Requirement 3: BIDS ordered descending, ASKS ordered ascending."""
    sp = StreamProcessor()
    sp.add_message({"seq": 1, "side": "BUY", "price": 10.0, "volume": 5})
    sp.add_message({"seq": 2, "side": "BUY", "price": 12.0, "volume": 5})
    sp.add_message({"seq": 3, "side": "SELL", "price": 20.0, "volume": 5})
    sp.add_message({"seq": 4, "side": "SELL", "price": 18.0, "volume": 5})

    depth = sp.get_book_depth()
    assert depth["BIDS"] == [(12.0, 5), (10.0, 5)]  # Correct: High to Low
    assert depth["ASKS"] == [(18.0, 5), (20.0, 5)]  # Correct: Low to High


def test_residual_volume():
    """
    Requirement 4: Any volume remaining after execution is added to the
    book at its specified price level.
    """
    sp = StreamProcessor()
    sp.add_message({"seq": 1, "side": "SELL", "price": 100.0, "volume": 10})

    # Incoming BUY 25 @ 100.0
    # Logic: 10 match against existing SELL, 15 remain as a new BID at 100.0
    sp.add_message({"seq": 2, "side": "BUY", "price": 100.0, "volume": 25})

    depth = sp.get_book_depth()
    assert depth["ASKS"] == []
    assert depth["BIDS"] == [(100.0, 15)]

ModuleNotFoundError: No module named 'solution'